In [ ]:
# Imports & Config
import json
import time
import hashlib
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
 
import pandas as pd
from google import genai
from google.genai import types

ModuleNotFoundError: No module named 'schema'

In [ ]:
# Paths
PROJECT_ROOT = Path("../../..").resolve()  # adjust based on notebook location
sys.path.insert(0, str(PROJECT_ROOT / "src" / "scripts"))
from schema.schema_validator import TriageSchemaValidator


DATA_DIR = Path("../data_collection/data/processed")
TRAIN_PATH = Path(DATA_DIR / "train.json")
VAL_PATH = Path(DATA_DIR / "val.json")
TEST_PATH = Path(DATA_DIR / "test.json")
OUTPUT_PATH = "results/gemini_flash_predictions.jsonl"
CHECKPOINT_PATH = "results/gemini_flash_checkpoint.jsonl"
SCHEMA_PATH = Path(PROJECT_ROOT / "src/schemas/triage_schema.json")
 
# ── Configuration ──────────────────────────────────────────────────────
load_dotenv(PROJECT_ROOT / ".env.local")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = "gemini-2.0-flash"

# Inference settings
MAX_BODY_CHARS = 8000       # truncate long issue bodies (~2k tokens)
REQUESTS_PER_MINUTE = 14    # stay just under the 15 RPM free-tier limit
N_SAMPLES = 1               # set to 3 or 5 later for majority-vote aggregation
TEMPERATURE = 0.0           # deterministic for N=1; raise to ~0.7 for N>1 sampling
 
client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
import json
with open(DATA_DIR / Path("test.json")) as f:
    data = json.load(f)
# Show keys and first record snippet
if isinstance(data, list):
    print(list(data[0].keys()))
elif isinstance(data, dict):
    print(list(data.keys())[:10])

['id', 'title', 'body', 'created_at', 'issue_type', 'platform', 'impact', 'topic', 'text_raw', 'text_clean']


In [4]:
## Schema & Label Vocabulary
# 
 
# %% Define label vocabulary and schema
with open(DATA_DIR / "label_vocab.json") as f:
    vocab = json.load(f)

COMPONENT_LABELS = vocab['components']
ISSUE_TYPES = vocab['issue_type']
PLATFORM_LABELS = vocab['platform']
IMPACT_LABELS = vocab['impact']
 
# Schema for reference (this is what the LLM must output)

with open(SCHEMA_PATH) as f:
    TRIAGE_SCHEMA = json.load(f)

print(f"Components ({len(COMPONENT_LABELS)}): {COMPONENT_LABELS}")
print(f"Issue types ({len(ISSUE_TYPES)}): {ISSUE_TYPES}")
print(f"Platforms ({len(PLATFORM_LABELS)}): {PLATFORM_LABELS}")
print(f"Impact ({len(IMPACT_LABELS)}): {IMPACT_LABELS}")

Components (18): ['2d', '3d', 'animation', 'buildsystem', 'core', 'dotnet', 'editor', 'export', 'gdscript', 'gui', 'import', 'input', 'other', 'physics', 'platforms', 'rendering', 'shaders', 'thirdparty']
Issue types (4): ['bug', 'discussion', 'docs', 'feature_request']
Platforms (5): ['android', 'linuxbsd', 'macos', 'web', 'windows']
Impact (4): ['crash', 'performance', 'regression', 'usability']


In [5]:
# Prompt Template 

 
SYSTEM_PROMPT = f"""You are an expert open-source issue triager for the Godot game engine.
 
Given a GitHub issue (title + body), produce a single JSON triage record.
 
## Output Schema
Return ONLY a valid JSON object with these fields — no markdown, no explanation:
 
{{
  "issue_type": one of {json.dumps(ISSUE_TYPES)},
  "components": list of applicable labels from {json.dumps(COMPONENT_LABELS)},
  "platform": list of applicable labels from {json.dumps(PLATFORM_LABELS)},
  "impact": list of applicable labels from {json.dumps(IMPACT_LABELS)},
  "needs_human_triage": true if you cannot confidently assign at least one component, else false,
  "confidence": float 0.0–1.0 indicating your overall confidence in this triage record
}}
 
## Rules
- "components" is the most important field. Assign 1–3 component labels that best match the issue.
- If the issue is ambiguous or you are unsure, set "needs_human_triage": true and "components": [].
- For "platform", infer from text cues (OS mentions, stack traces, etc.). Use an empty list [] if unclear.
- For "impact", only include labels if there is clear textual evidence (e.g., "crash", "regression from 4.x").
- If no impact labels apply, return an empty list.
- Return ONLY the JSON object. No other text.
"""
 
 
def build_user_prompt(title: str, body: str) -> str:
    """Construct the user message from issue title and body."""
    if pd.isna(body):
        body = ""
    if pd.isna(title):
        title = ""
    # Truncate long bodies to stay within context budget
    if len(body) > MAX_BODY_CHARS:
        body = body[:MAX_BODY_CHARS] + "\n\n[... truncated ...]"
    return f"## Issue Title\n{title}\n\n## Issue Body\n{body}"

In [6]:
def call_gemini(title: str, body: str, retries: int = 3) -> dict:
    """
    Send a single issue to Gemini and return a parsed triage record.
    Includes retry logic for malformed JSON and API errors.
    """
    user_prompt = build_user_prompt(title, body)
 
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    response_mime_type="application/json",
                    temperature=TEMPERATURE,
                ),
            )
            raw_text = response.text.strip()
            record = json.loads(raw_text)
 
            # Validate and clamp to allowed vocabularies
            validated = validate_record(record)
            return validated
 
        except json.JSONDecodeError as e:
            print(f"  [attempt {attempt+1}] JSON parse error: {e}")
            if attempt < retries - 1:
                time.sleep(2)
            else:
                return make_fallback_record(reason=f"json_error: {e}")
 
        except Exception as e:
            error_str = str(e)
            print(f"  [attempt {attempt+1}] API error: {error_str}")
            if "429" in error_str or "quota" in error_str.lower():
                wait = 30 * (attempt + 1)
                print(f"  Rate limited. Waiting {wait}s...")
                time.sleep(wait)
            elif attempt < retries - 1:
                time.sleep(5)
            else:
                return make_fallback_record(reason=f"api_error: {error_str}")
 
    return make_fallback_record(reason="max_retries_exceeded")

In [7]:
def clamp_to_vocab(record: dict) -> dict:
    """
    Pre-validation step: filter predicted values to allowed vocabularies.
    Catches hallucinated labels before schema validation.
    """
    clamped = {}
    
    it = record.get("issue_type", "bug")
    clamped["issue_type"] = it if it in ISSUE_TYPES else "bug"
    
    comps = record.get("components", [])
    clamped["components"] = [c for c in comps if c in COMPONENT_LABELS]
    
    plats = record.get("platform", [])
    clamped["platform"] = [p for p in plats if p in PLATFORM_LABELS]
    
    imps = record.get("impact", [])
    clamped["impact"] = [i for i in imps if i in IMPACT_LABELS]
    
    nht = record.get("needs_human_triage", False)
    if not clamped["components"]:
        nht = True
    clamped["needs_human_triage"] = bool(nht)
    
    conf = record.get("confidence", 0.0)
    clamped["confidence"] = max(0.0, min(1.0, float(conf)))
    
    return clamped

In [ ]:
def validate_record(record: dict) -> tuple[dict, bool, list]:
    """
    Clamp to vocab, then validate against the JSON schema.
    Returns (record, schema_valid, errors).
    """
    clamped = clamp_to_vocab(record)
    ok, obj, errors = schema_validator.validate_instance(clamped)
    return clamped, ok, errors

In [ ]:
def make_fallback_record(reason: str = "unknown") -> dict:
    """Return a safe fallback record when inference fails."""
    return {
        "issue_type": "bug",
        "components": [],
        "platform": [],
        "impact": [],
        "needs_human_triage": True,
        "confidence": 0.0,
        "_error": reason,
    }